# Your score peaked early because you were meeting weak opponents

I shipped four agents in two days. Every one of them did the same thing: climbed fast, peaked somewhere over 2000 in the first two hours, then slid back and flattened out near 1890.

I read the first peak as the agent's strength and the slide as bad luck. It is the other way round. A new submission starts near 600 and gets matched with whatever is down there, so the early climb is mostly a statement about who it played, not how well.

The fix is to stop reading the score and read the win rate **bucketed by opponent rating**. When I did that, three agents that looked different turned out to be the same agent, and one that looked ordinary turned out to be the only real improvement I had made.

## What the four submissions looked like

| submission | games | peak score | settled at | overall win rate | mean opponent |
|---|---|---|---|---|---|
| A | 45 | 1746 | 1738 | **71%** | **1535** |
| B | 97 | 2024 (game 15) | 1894 | 56% | 1816 |
| C | 111 | 2069 (game 21) | 1887 | 53% | 1824 |
| D | 86 | 2094 | 1890 | 57% | 1768 |

Submission A has by far the best win rate and by far the worst score. That is the whole thing in one row: it won 71% of its games because its opponents averaged 1535, and it never climbed high enough to meet anyone harder.

The peaks are not strength either. They land between game 15 and game 21 in every case, which is exactly where the Elo has climbed far enough to start drawing real opposition.

## The same four, bucketed

Now against opponents rated 1750 and up, which is where these agents actually live:

| submission | vs opponents >= 1750 |
|---|---|
| B | 43/86 = **50%** |
| C | 46/99 = **46%** |
| D | 34/71 = **48%** |

Three agents, three very different offline benchmarks, one number. B, C and D are the same agent as far as the ladder is concerned, and I had spent a day and a half telling myself D was better because my offline gate said +28 wins per 200 games.

A fifth submission, built on a different idea, currently reads **13/15 = 87%** against the same bucket on a small sample. That is the only one of the five where the bucket moved, and it is the one I would have dismissed if I had waited for its score, because its score is still low — its opponents so far average 1535, the same place submission A was stuck.

## Where the games actually are

Pooling 173 real episodes from two submissions and bucketing by the opponent's rating going in:

| opponent rating | games | my win rate | mean coin margin |
|---|---|---|---|
| under 1500 | 18 | **100%** | +35,495 |
| 1500 - 1750 | 5 | **100%** | +10,230 |
| **1750 - 2000** | **141 (82%)** | **51%** | **-670** |
| over 2000 | 9 | 11% | -6,015 |

82% of my games sit in one band, and in that band I am a coin flip. Everything above and below it is decided before the game starts.

The margin column is the useful part. In the band that matters the median margin is **+167 coins** on a 100,000-coin game. Those are not games I am losing badly; they are games decided by a rounding error, which is a much more encouraging picture than the score suggested, and it tells you the size of edge worth chasing.

## Why the score misleads in a rating system

Three things stack up:

**A new submission starts near the bottom.** Mine entered at 698, 708 and 715. Everything it plays for the first hour is below where it belongs.

**Rating gain is largest when you are underrated.** So the early climb is fast and looks like a trend. It is a correction.

**Then the matching catches up.** Once you are at your level you play your level, your win rate falls to about 50%, and the curve flattens or drifts down. Every one of my four did this between game 15 and game 25.

None of that is a flaw in the leaderboard. It is what a rating is supposed to do. It just means the number is not a measurement you can read early, and the peak is never the thing to quote.

## The read that works

Pull your own episodes, bucket by the opponent's rating at the start of the game, and compare submissions inside the bucket where most of your games are. That is a like-for-like comparison; the raw score is not, because two submissions of the same age have met different fields.

The episode list gives you everything you need per game: both agents' submission ids, the reward each got, and the opponent's rating going in.

The function below takes `(opponent_rating, i_won, my_coins - their_coins)` triples and prints the table above. Feed it your own and see whether the submission you are proud of actually moved the band you live in.

In [ ]:
from statistics import mean, median

BANDS = [(0, 1500), (1500, 1750), (1750, 2000), (2000, 2250), (2250, 10**9)]


def by_band(games, bands=BANDS):
    """games: iterable of (opponent_rating, won, coin_margin).

    Prints win rate per opponent-strength band. Compare two submissions inside the
    band that holds most of your games; the raw public score compares two different
    fields and cannot be read that way.
    """
    total = len(games)
    print(f"{'opponent rating':>18}{'games':>8}{'share':>8}{'win rate':>10}{'median margin':>16}")
    rows = []
    for lo, hi in bands:
        sel = [g for g in games if lo <= g[0] < hi]
        if not sel:
            continue
        w = sum(1 for g in sel if g[1])
        rows.append((lo, hi, len(sel), w, median(g[2] for g in sel)))
        label = f"{lo}-{hi}" if hi < 10**9 else f"{lo}+"
        print(f"{label:>18}{len(sel):>8}{100*len(sel)/total:>7.0f}%"
              f"{100*w/len(sel):>9.0f}%{rows[-1][4]:>16,.0f}")
    big = max(rows, key=lambda r: r[2])
    print()
    print(f"{100*big[2]/total:.0f}% of games are in the {big[0]}-{big[1]} band, "
          f"where you win {100*big[3]/big[2]:.0f}%.")
    print("That is the number to compare submissions on. Not the score, and never the peak.")


# a synthetic stand-in with the shape I measured, so the cell runs without the API:
# lopsided at both ends, a coin flip in the middle, and the middle holds most games.
import random
rng = random.Random(23)
games = ([(rng.uniform(1200, 1500), True, rng.uniform(20000, 50000)) for _ in range(18)]
         + [(rng.uniform(1500, 1750), True, rng.uniform(5000, 15000)) for _ in range(5)]
         + [(rng.uniform(1750, 2000), rng.random() < 0.51, rng.gauss(-670, 6000)) for _ in range(141)]
         + [(rng.uniform(2000, 2200), rng.random() < 0.11, rng.gauss(-6015, 4000)) for _ in range(9)])
by_band(games)

# To use your own, list your episodes and build the triples:
#   POST https://www.kaggle.com/api/i/competitions.EpisodeService/ListEpisodes
#        {"submissionId": <your submission id>}
# each episode carries both agents with `reward` and `initialScore`; take the agent
# whose submissionId is NOT yours as the opponent.

## The short version

The early peak is the rating correcting, not the agent performing. Mine peaked between game 15 and 21 every single time and then gave most of it back.

Compare submissions by win rate inside the band that holds most of your games. Three of mine looked different on four different offline benchmarks and were 50%, 46% and 48% where it counted.

And look at the margin, not only the win. Mine is a coin flip in that band with a median margin of 167 coins, which says the games are there to be taken and tells you how small an edge would do it.

If you have a submission whose bucketed win rate moved but whose score did not, or the other way round, I would like to see it. I only have five, and four of them said the same thing.